# 排名融合与复现检查

负责人：C主实现，A/B独立核对。这是中文教学参考，正式实现由成员理解后编写、执行并核对。

输入挂载：官方数据、共享Dataset、三位成员保存的baseline/neighborhood/hybrid输出目录；最后可额外挂载历史两模型输出作核对。无需挂载旧track代码包。CPU训练，四线程；机器等待另计。

本项目自训练预测组合。40/60是历史冻结部署比例，50/50是对照。已有OOF上的选权属于开发评估，基础OOF训练集存在交叉依赖，不能称严格端到端nested CV。


- [比赛数据与规则](https://www.kaggle.com/competitions/playground-series-s6e9)
- [LightGBM论文](https://proceedings.neurips.cc/paper/2017/hash/6449f44a102fde848669bdd9eb6b76fa-Abstract.html)
- [目标编码：内部交叉拟合与平滑](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.TargetEncoder.html)
- [AUC定义](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)

本教程生成时尚未执行Kaggle完整训练。历史分数是核对参照，不是本轮结果。阅读当前文档不意味着升级历史环境。

## 如何学习本文件

每次只运行一个单元，先用自己的话预测输出。`iloc`按位置取行，`loc`按标签取行；`to_numpy`去掉索引，之后必须保证位置对应。`assert`是验收条件，失败应查数据而非删除检查。`fit`从数据学习，`transform`使用已学习规则。

编程练习：修改一个小例子的输入并解释变化；正式配置保持历史定义。复杂特征组的整体增益不能归因于单一列。

## 读取数据及共享分组

先统一行序，再组合分数。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import json
import gc
import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

INPUT = Path('/kaggle/input')
TARGET = 'Will_Buy_EV'
SEED = 42
N_SPLITS = 5

def unique_file(name):
    """从已挂载输入中定位唯一文件；多个版本时停止，防止静默读错。"""
    paths = list(INPUT.rglob(name))
    assert len(paths) == 1, f'Expected one {name}, found {paths}'
    return paths[0]

competition_dirs = [INPUT/'competitions/playground-series-s6e9', INPUT/'playground-series-s6e9']
available = [p for p in competition_dirs if (p/'train.csv').is_file()]
assert len(available) == 1, 'Attach the official competition data.'
DATA = available[0]
train = pd.read_csv(DATA/'train.csv')
test = pd.read_csv(DATA/'test.csv')
sample = pd.read_csv(DATA/'sample_submission.csv')
y = train[TARGET].map({'No':0, 'Yes':1})
assert y.notna().all() and set(y.unique()) == {0,1}
assert train.id.is_unique and test.id.is_unique
assert sample.columns.tolist() == ['id',TARGET] and sample.id.equals(test.id)
assert train.columns.drop(['id',TARGET]).tolist() == test.columns.drop('id').tolist()

def align_rows(frame, ids):
    """先检查一一对应，再按官方顺序排列；不能直接假设CSV行序相同。"""
    assert frame.id.is_unique and len(frame) == len(ids)
    assert set(frame.id) == set(ids)
    return frame.set_index('id').loc[ids].reset_index()

def current_versions():
    return {'lightgbm':lgb.__version__, 'sklearn':sklearn.__version__,
            'numpy':np.__version__, 'pandas':pd.__version__}


fold_path = unique_file('shared_folds.csv')
shared = align_rows(pd.read_csv(fold_path), train.id)
assert np.array_equal(shared.target, y)
assert shared.fold.notna().all() and shared.fold.isin(range(5)).all()
assert set(shared.fold) == set(range(5))
fold_ids = shared.fold.to_numpy(dtype=int)
foundation_note = json.loads((fold_path.parent/'dataset_note.json').read_text())
display(shared.groupby('fold').agg(rows=('id','size'),positive_rate=('target','mean')))
print(current_versions())

## 定位三个新运行并严格对齐

通过summary里的描述性run_name选择输出。历史文件没有这些新名称，不会替代新训练。重复挂载多个运行时明确停止，由成员选择唯一版本。

In [ ]:
def read_new_run(name):
    directories=[]
    for p in INPUT.rglob('run_summary.json'):
        metadata=json.loads(p.read_text())
        if metadata.get('run_name')==name:
            directories.append(p.parent)
    assert len(directories)==1, (name,directories)
    directory=directories[0]
    oof=align_rows(pd.read_csv(directory/'oof_predictions.csv'),train.id)
    sub=align_rows(pd.read_csv(directory/'submission.csv'),test.id)
    assert np.array_equal(oof.target,y) and np.array_equal(oof.fold,fold_ids)
    assert oof.prediction.between(0,1).all() and sub[TARGET].between(0,1).all()
    return oof.prediction.to_numpy(),sub[TARGET].to_numpy(),str(directory)
runs={name:read_new_run(name) for name in ['baseline','neighborhood','hybrid']}

## 理解全局排名

[0.2,0.8,0.8]的平均并列名次是[1,2.5,2.5]。除以3得到百分位排名。这里使用整个OOF的排名，不能groupby(fold)后排名。排名融合分数并非校准后的购买概率。

In [ ]:
from scipy.stats import rankdata
def percentile_rank(values):
    """输入有限的一维预测；输出保留并列关系的全局百分位排名。"""
    values=np.asarray(values,dtype=float)
    assert values.ndim==1 and len(values)>0 and np.isfinite(values).all()
    return rankdata(values,method='average')/len(values)
assert np.allclose(percentile_rank([0.2,0.8,0.8]),[1/3,2.5/3,2.5/3])
left,right=percentile_rank(runs['neighborhood'][0]),percentile_rank(runs['hybrid'][0])
print('Rank correlation:',np.corrcoef(left,right)[0,1])

## 复现选权并计算固定融合

先计算全局排名，再切四折开发集选择混合特征模型的权重；41个候选间隔0.025。AUC相同时选择靠近0.5的权重，仍相同则选择较小权重以保证确定性。每折留出评价仍属于已有OOF上的开发检查。最终提交保持历史40/60，新中位数另行记录。

In [ ]:
selection=[]
selected_oof=np.empty(len(train))
for fold in range(5):
    development=fold_ids!=fold
    held_out=~development
    scores=[(float(w),float(roc_auc_score(y[development],(1-w)*left[development]+w*right[development]))) for w in np.linspace(0,1,41)]
    weight,score=min(scores,key=lambda item:(-item[1],abs(item[0]-0.5),item[0]))
    selected_oof[held_out]=(1-weight)*left[held_out]+weight*right[held_out]
    selection.append({'held_out_fold':fold,'hybrid_weight':weight,'development_auc':score,
                      'held_out_auc':float(roc_auc_score(y[held_out],selected_oof[held_out]))})
median_weight=float(np.median([r['hybrid_weight'] for r in selection]))
candidates={name:values[0] for name,values in runs.items()}
candidates.update(equal_rank=0.5*left+0.5*right,frozen_rank=0.4*left+0.6*right,
                  selected_weights=selected_oof,median_rank=(1-median_weight)*left+median_weight*right)
overall={name:float(roc_auc_score(y,pred)) for name,pred in candidates.items()}
fold_results=[{'fold':fold,**{name:float(roc_auc_score(y[fold_ids==fold],pred[fold_ids==fold])) for name,pred in candidates.items()}} for fold in range(5)]
display(pd.DataFrame(selection))
display(pd.DataFrame(fold_results))
print(overall,'New median hybrid weight:',median_weight)

## 保存40/60提交与复现证据

两份输入submission已经各自平均五折概率，直接对各自测试预测排名后加权。保存一份最终提交，不自动上传。权重表放入summary，避免重复文件。

In [ ]:
output=Path('/kaggle/working/final_rank_blend')
output.mkdir(exist_ok=False)
final_test=0.4*percentile_rank(runs['neighborhood'][1])+0.6*percentile_rank(runs['hybrid'][1])
submission=sample.copy()
submission[TARGET]=final_test
assert submission.id.equals(test.id) and submission[TARGET].between(0,1).all()
submission.to_csv(output/'submission.csv',index=False)
pd.DataFrame({'id':train.id,'target':y,'fold':fold_ids,'prediction':candidates['frozen_rank']}).to_csv(output/'oof_predictions.csv',index=False)
report=(f'Two newly trained LightGBM models were combined using global percentile ranks and historical frozen weights 0.40 and 0.60. '
        f'The frozen blend OOF ROC AUC was {overall["frozen_rank"]:.9f}; the equal-weight control was {overall["equal_rank"]:.9f}. '
        'Weight selection over existing OOF predictions is a development analysis, not end-to-end nested cross-validation. '
        'Public and private performance for the new submission remains unverified.')
summary={'weights':{'neighborhood':0.4,'hybrid':0.6},'new_median_hybrid_weight':median_weight,
         'input_directories':{k:v[2] for k,v in runs.items()},'oof_metrics':overall,'fold_metrics':fold_results,
         'weight_selection':selection,'versions':current_versions(),'public_score':None,'report_summary':report}
(output/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
print(report)
print(output)

## 核对两套历史预测

最终提交前挂载数据说明中列出的两个历史输出。按ID比较最大绝对差、平均绝对差及AUC，同时对齐特征列和环境。历史文件缺失则明确报错，不能声称完成逐行一致性验收。原最终提交文件不在本包内；下面重建历史40/60文件作数值对照。

In [ ]:
checks=[]
historical_test=[]
for name in ['neighborhood','hybrid']:
    directory=Path(foundation_note['history_directories'][name])
    assert directory.is_dir(), f'Attach historical output: {directory}'
    old=align_rows(pd.read_csv(directory/'oof_predictions.csv'),train.id)
    old_sub=align_rows(pd.read_csv(directory/'submission.csv'),test.id)
    assert np.array_equal(old.target,y) and np.array_equal(old.fold,fold_ids)
    column='probability' if 'probability' in old else 'prediction'
    difference=np.abs(runs[name][0]-old[column].to_numpy())
    test_difference=np.abs(runs[name][1]-old_sub[TARGET].to_numpy())
    historical_test.append(old_sub[TARGET].to_numpy())
    checks.append({'model':name,'old_auc':float(roc_auc_score(y,old[column])),
                   'new_auc':overall[name],'max_oof_difference':float(difference.max()),
                   'mean_oof_difference':float(difference.mean()),'max_test_difference':float(test_difference.max())})
old_blend=0.4*percentile_rank(historical_test[0])+0.6*percentile_rank(historical_test[1])
summary['reproduction_checks']=checks
summary['max_final_difference_vs_reconstructed_history']=float(np.max(np.abs(final_test-old_blend)))
(output/'run_summary.json').write_text(json.dumps(summary,indent=2,allow_nan=False),encoding='utf-8')
display(pd.DataFrame(checks))
print('Final max difference:',summary['max_final_difference_vs_reconstructed_history'])

## 分析与交接

历史参照：两单模OOF为0.946046626、0.946129123，50/50为0.946192671，40/60为0.946196333，历史公开榜0.94642。实际不一致先查输入版本、fold、列顺序、数据类型、编码和参数。接近或相同OOF不保证测试预测相同。

A审核后手动提交保存的submission，并把真实成绩更新到summary；私人榜未公布就写未知。三人分别解释：B说明编码标签范围，A说明邻域统计，C说明排名和选权。

来源：[排名文档](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.rankdata.html)、[嵌套验证说明](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)。历史Notebook22的诊断使用(rank-0.5)/n，交接文档的最终文件使用rank/n；固定同一权重和样本数时二者相差共同常数，AUC不变，但文件数值不同。本文件遵循最终交接文档rank/n。